In [0]:
%sql
-- ============================================================================
-- MPSII DIAGNOSED PATIENT COHORT TABLE (SIMPLIFIED OUTPUT)
-- ============================================================================
-- Output: PATIENT_ID, AGE, AGE_GROUP, GENDER, COHORT
-- Time Window: August 1, 2015 - July 31, 2025
-- Treatment Codes: C9232, J1743 only
-- Diagnosis Codes: E76.1 only (E76.3 excluded)
-- ============================================================================

CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.MPSII_DIAGNOSED AS

WITH diagnosis_claims AS (
    -- Extract all E76.1 diagnosis claims from medical events
    SELECT DISTINCT 
        PATIENT_ID,
        SERVICE_DATE AS CLAIM_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2015-08-01' AND '2025-07-31'
    
    UNION
    
    -- Extract all E76.1 diagnosis claims from pharmacy events
    SELECT DISTINCT 
        PATIENT_ID,
        FILL_DATE AS CLAIM_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2015-08-01' AND '2025-07-31'
),

diagnosis_counts AS (
    -- Count distinct diagnosis dates per patient
    SELECT 
        PATIENT_ID,
        COUNT(DISTINCT CLAIM_DATE) AS dx_claim_count
    FROM diagnosis_claims
    GROUP BY PATIENT_ID
),

treatment_claims AS (
    -- Extract treatment claims (C9232 and J1743 procedure codes)
    SELECT DISTINCT 
        PATIENT_ID,
        SERVICE_DATE AS CLAIM_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN ('C9232', 'J1743')
      AND SERVICE_DATE BETWEEN '2015-08-01' AND '2025-07-31'
),

treatment_counts AS (
    -- Count distinct treatment dates per patient
    SELECT 
        PATIENT_ID,
        COUNT(DISTINCT CLAIM_DATE) AS tx_claim_count
    FROM treatment_claims
    GROUP BY PATIENT_ID
),

patient_demographics AS (
    -- Get patient demographics with standardized gender values
    SELECT 
        PATIENT_ID,
        AGE,
        CASE 
            WHEN AGE <= 5 THEN '≤5'
            ELSE '>5'
        END AS AGE_GROUP,
        CASE 
            WHEN UPPER(PATIENT_GENDER) IN ('M', 'MALE') THEN 'M'
            WHEN UPPER(PATIENT_GENDER) IN ('F', 'FEMALE') THEN 'F'
            ELSE PATIENT_GENDER
        END AS GENDER
    FROM (
        SELECT 
            PATIENT_ID,
            PATIENT_GENDER,
            (2025 - YEAR(PATIENT_YOB)) AS AGE
        FROM com_edp_prd.com_raw.kom_patient_demographics
    )
),

patient_clinical_data AS (
    -- Combine diagnosis and treatment counts
    SELECT 
        COALESCE(d.PATIENT_ID, t.PATIENT_ID) AS PATIENT_ID,
        COALESCE(d.dx_claim_count, 0) AS dx_claim_count,
        COALESCE(t.tx_claim_count, 0) AS tx_claim_count
    FROM diagnosis_counts d
    FULL OUTER JOIN treatment_counts t
        ON d.PATIENT_ID = t.PATIENT_ID
)

-- Final cohort assignment
SELECT 
    dem.PATIENT_ID,
    dem.AGE,
    dem.AGE_GROUP,
    dem.GENDER,
    CASE 
        -- Younger Male: Male, Age ≤5, (2+ Dx OR 1+ Dx + 2+ Tx)
        WHEN dem.GENDER = 'M' 
            AND dem.AGE <= 5 
            AND (clin.dx_claim_count >= 2 OR (clin.dx_claim_count >= 1 AND clin.tx_claim_count >= 2))
        THEN 'Younger Male'
        
        -- Older Male: Male, Age >5, (20+ Dx OR 1+ Dx + 2+ Tx)
        WHEN dem.GENDER = 'M' 
            AND dem.AGE > 5 
            AND (clin.dx_claim_count >= 20 OR (clin.dx_claim_count >= 1 AND clin.tx_claim_count >= 2))
        THEN 'Older Male'
        
        -- Female: Female, Any Age, (20+ Dx OR 1+ Dx + 2+ Tx)
        WHEN dem.GENDER = 'F' 
            AND (clin.dx_claim_count >= 20 OR (clin.dx_claim_count >= 1 AND clin.tx_claim_count >= 2))
        THEN 'Female'
        
        ELSE NULL
    END AS COHORT
FROM patient_demographics dem
INNER JOIN patient_clinical_data clin
    ON dem.PATIENT_ID = clin.PATIENT_ID
WHERE clin.dx_claim_count >= 1  -- Must have at least 1 E76.1 diagnosis
  AND CASE 
        WHEN dem.GENDER = 'M' AND dem.AGE <= 5 
            AND (clin.dx_claim_count >= 2 OR (clin.dx_claim_count >= 1 AND clin.tx_claim_count >= 2))
        THEN 1
        WHEN dem.GENDER = 'M' AND dem.AGE > 5 
            AND (clin.dx_claim_count >= 20 OR (clin.dx_claim_count >= 1 AND clin.tx_claim_count >= 2))
        THEN 1
        WHEN dem.GENDER = 'F' 
            AND (clin.dx_claim_count >= 20 OR (clin.dx_claim_count >= 1 AND clin.tx_claim_count >= 2))
        THEN 1
        ELSE 0
      END = 1  -- Only include patients who qualify for a cohort
ORDER BY COHORT, AGE, PATIENT_ID;